[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NguyenVu04/band-tilt/blob/main/notebooks/00_scenario.ipynb)

# 00 — Scenario Preparation

**Purpose.** Assemble one *scenario*: the 3D environment, its radio materials,
the road network, the cell-band table, the current tilts, the per-cell-band tilt
bounds and the band-priority weights. PROJECT.md section 16, Phase 1.

A scenario is the unit everything downstream is indexed by. It is what
notebook 01 populates with UEs, what notebook 02 ray-traces, what notebook 03
splits on, and what section 12 perturbs to ask whether any of this generalises.
Getting its definition wrong is expensive later and cheap to catch here.

**Inputs.** `data/external/simulation_map/`, `data/raw/gcell_conf.csv`,
`configs/radio.yaml`.

**Outputs.** The cell-band table and its row order — the canonical ordering of
every tilt vector in the project — plus a scenario manifest.

**Requires** `uv sync --extra rt`.

> **Blocked.** The multi-band cell configuration has not arrived, so the band
> dimension of the table cannot be built from real data and `configs/radio.yaml`
> still holds placeholders. Sections 5 and 6 below quantify exactly what is
> missing rather than inventing it.

## 0. Environment

Run this section first, wherever you are.

**Locally** it only walks up to the project root and makes it the working
directory, so the root-relative paths in `configs/data.yaml` resolve the same way
they do for `task clean:data` and the DVC pipeline. Nothing is installed.

**In Colab** it also clones the repository, puts it on `sys.path` so `import src`
works without an editable install, and installs the packages Colab does not ship.
Note that `data/` and `models/` are DVC-tracked and therefore *not* part of the
clone — a fresh runtime has neither. See the Drive cell below.

In [ ]:
# --- Environment bootstrap -------------------------------------------------
# Identical in every notebook except the COLAB_PACKAGES line below, which names
# the extras this particular notebook needs. Forked the repository? Change these
# three values and the badge URL at the top of this notebook.
REPO_URL = "https://github.com/NguyenVu04/band-tilt.git"
BRANCH = "main"
SUBDIR = ""  # the project root is the repository root

# (import name, pip name). Colab already ships numpy, pandas, pyarrow,
# scikit-learn, joblib, matplotlib and seaborn, so only these are installed —
# which keeps the bootstrap fast and avoids a "restart runtime" prompt.
COLAB_PACKAGES = [("hydra", "hydra-core"), ("sionna_rt", "sionna-rt")]

import importlib.util
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    checkout = Path("/content") / Path(REPO_URL).stem
    if not checkout.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(checkout)],
            check=True,
        )
    root = checkout / SUBDIR
    missing = [pip for mod, pip in COLAB_PACKAGES if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    # JupyterLab starts the kernel in notebooks/; walk up to the project root.
    root = Path.cwd()
    while not (root / "pyproject.toml").exists() and root != root.parent:
        root = root.parent

os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))  # makes `import src` work without an editable install

# Extra Hydra overrides consumed by load_config() in section 1. Empty unless
# the Drive cell below fills it in, so local runs are unaffected.
CONFIG_OVERRIDES: list[str] = []

print(f"project root: {root}   colab: {IN_COLAB}")

In [ ]:
# --- Colab: data and artifacts (optional) ----------------------------------
# data/ and models/ are DVC-tracked, so they are not in the Git clone and a
# fresh Colab runtime has neither. Mount Drive and point the config at it —
# Drive also survives a runtime reset, which /content does not.
#
# The scene is the large one: data/external/simulation_map/ holds 3,753 meshes,
# so keep it on Drive rather than re-downloading it per session.
#
# from google.colab import drive
#
# drive.mount("/content/drive")
# DATA_ROOT = "/content/drive/MyDrive/band-tilt/data"
# CONFIG_OVERRIDES += [
#     f"data.mdt_path={DATA_ROOT}/raw/measurement_data.csv",
#     f"data.cell_config_path={DATA_ROOT}/raw/gcell_conf.csv",
#     f"data.scene_file={DATA_ROOT}/external/simulation_map/scene.xml",
#     f"data.train_path={DATA_ROOT}/processed/mdt_train.parquet",
#     f"data.test_path={DATA_ROOT}/processed/mdt_test.parquet",
# ]

## 1. Setup

Compose the config, seed everything, and import from `src/`. Every notebook
starts the same way so that a cell copied between notebooks behaves identically.

In [ ]:
# Standard setup for every notebook in this project.
# Autoreload so edits in src/ take effect without restarting the kernel.
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.config import load_config
from src.utils.plotting import setup_plotting
from src.utils.seed import set_seed

cfg = load_config(overrides=CONFIG_OVERRIDES)
set_seed(cfg.seed)
setup_plotting()  # matplotlib/seaborn styling for report-ready figures

pd.set_option("display.max_columns", 50)
cfg

## 2. The cell configuration as it stands

One row per gcell. Check what is actually in the export before assuming the
table can be built from it.

In [ ]:
from src.data.load import load_cell_config
from src.radio import cell_band, scene

cells_df = load_cell_config(cfg)
print(f"{len(cells_df)} cells across {cells_df.gnodeb_id.nunique()} sites")
print(f"columns: {list(cells_df.columns)}")
cells_df.head()

## 3. Sites, sectors and azimuths

Cells per site, and the azimuth spread within a site. Three sectors at roughly
120° is the usual pattern; anything else changes what "neighbouring cell-band"
means and therefore what overlap rate measures.

In [ ]:
per_site = cells_df.groupby("gnodeb_id").size()
print(per_site.describe())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
per_site.value_counts().sort_index().plot.bar(ax=axes[0])
axes[0].set(xlabel="cells per site", ylabel="sites", title="Sector count")
axes[1].hist(cells_df["azimuth"], bins=36)
axes[1].set(xlabel="azimuth (deg)", ylabel="cells", title="Azimuth distribution")
plt.tight_layout()

## 4. Scene extent and coordinate alignment

The cells must sit inside the scene, in the same local metric frame. A cell
outside the bounding box is either a coordinate-frame mismatch or a cell that
should not be in the scenario — both are worth knowing before a solve.

In [ ]:
xmin, ymin, xmax, ymax = scene.scene_bounds(cfg)
print(f"scene extent: x [{xmin}, {xmax}]  y [{ymin}, {ymax}]")

inside = (
    cells_df.sim_x.between(xmin, xmax) & cells_df.sim_y.between(ymin, ymax)
)
print(f"cells inside the scene: {inside.sum()} / {len(cells_df)}")

fig, ax = plt.subplots(figsize=(7, 7))
ax.add_patch(plt.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin, fill=False, lw=1.5))
ax.scatter(cells_df.sim_x, cells_df.sim_y, s=28, marker="^")
ax.set(xlabel="sim_x (m)", ylabel="sim_y (m)", title="Cells in the scene frame")
ax.set_aspect("equal")

## 5. The band dimension — what is missing

PROJECT.md section 8 requires band, carrier frequency, transmit power and an
electrical/mechanical tilt split per cell-band. The current export has none of
them: it carries one `digital_tilt` per cell and no band column at all.

This is the project's blocking gap. **Do not fabricate band data to make
something run** — a plausible-looking band table produces a plausible-looking
radio map and an entirely fictional result.

In [ ]:
pending = [c for c in cfg.data.schema.cell_config.columns if c.startswith("<")]
present = [c for c in cfg.data.schema.cell_config.columns if not c.startswith("<")]

print("declared and present :", present)
print("declared and PENDING :", pending)
print()
print("in the export        :", sorted(cells_df.columns))
missing = [c for c in present if c not in cells_df.columns]
print("declared but absent  :", missing or "none")

## 6. Band table and tilt bounds

`configs/radio.yaml` is the only place the band set is declared. Its
`priority_weight` is `w_b` for the Band Priority Score (PROJECT.md section 4.7),
and `tilt.min` / `tilt.max` bound the decision variable itself — these two
numbers *are* the feasible set `X` (section 3.2).

Placeholders here are deliberate. `src.config.validate_config` is the guard that
stops one reaching a numeric call site.

In [ ]:
from omegaconf import OmegaConf

bands = OmegaConf.to_container(cfg.radio.bands, resolve=False)
band_df = pd.DataFrame(bands).T
band_df.index.name = "band_id"
print(f"{len(band_df)} band(s) declared")
band_df

## 7. Build the cell-band table

The atomic unit of the decision variable. Its row order is the canonical
ordering of every tilt vector in the project — record it with any result you
save, or the numbers cannot be mapped back to cells later.

This is the cell that will fail until the multi-band export lands.

In [ ]:
table = cell_band.build_table(cells_df, cfg)

tilt_0 = cell_band.current_tilt(table)
lower, upper = cell_band.tilt_bounds(table)
print(f"{len(table)} cell-band pairs = {table.gcell_id.nunique()} cells x {table.band.nunique()} bands")
print(f"search space X has {len(table)} dimensions")
table.head(10)

## 8. Radio materials

PROJECT.md section 7.2. Surfaces carry permittivity, conductivity and scattering
parameters, and those drive reflection and diffraction as much as the geometry
does. They are also one of the things section 12 perturbs between scenarios, so
record what this scenario uses rather than accepting the loader's defaults
silently.

In [ ]:
sc = scene.load_scene(cfg)
# TODO(1): enumerate the scene's materials and their radio properties
# TODO(2): flag every surface still carrying a library default
# TODO(3): record the material assignment in the scenario manifest below
print(f"scene loaded: {sc}")

## 9. Scenario manifest

Everything above, written down as one record. PROJECT.md section 22.4 asks for
reproducible evaluation: a scenario identifier, the seeds, the config version
and the model version stored with every output.

`scenario_id` is what `configs/data.yaml` splits on (section 12.3), so it has to
exist before notebook 03 can hold anything out.

In [ ]:
import json

manifest = {
    "scenario_id": "<scenario identifier — see the gap note below>",
    "scene_file": cfg.data.scene_file,
    "scene_bounds": [xmin, ymin, xmax, ymax],
    "n_cells": int(cells_df.gcell_id.nunique()),
    "n_cell_bands": int(len(table)),
    "bands": list(band_df.index),
    "grid": OmegaConf.to_container(cfg.radio.grid, resolve=True),
    "ray_tracing": OmegaConf.to_container(cfg.radio.ray_tracing, resolve=False),
    "seed": cfg.seed,
}
print(json.dumps(manifest, indent=2, default=str))

## 10. Findings — what 01 and 02 need from here

Fill this in from the cells above; it is the handoff, and the numbers belong in
the text rather than in a cell someone has to re-run.

| Question | Answer |
|---|---|
| Cells, sites, sectors | |
| Cell-band pairs (dimensions of `X`) | |
| Scene extent, and cells outside it | |
| Bands declared, and their weights | |
| Tilt bounds per band | |
| Materials still on library defaults | |

**Handoff checklist**

- [ ] Every cell sits inside the scene bounding box, in the same metric frame.
- [ ] The cell-band table builds, and its row order is recorded in the manifest.
- [ ] `cfg.radio.grid.cell_size_m` is fixed. It changes every reported KPI, so
      it must be settled before the first surrogate sample is generated.
- [ ] `cfg.radio.ray_tracing` is fixed, for the same reason.
- [ ] Band priority weights are ordered as intended — `w_low < w_mid < w_high`
      is the spec's design direction, not a constraint the code enforces.
- [ ] `scenario_id` is defined and will be carried on every record downstream.

**Known gaps carried forward**

- The multi-band cell configuration has not arrived; sections 5 and 7 fail
  until it does.
- `scenario_id` has no producer yet. Scenario-level splitting (PROJECT.md
  section 12.3) depends on it, and so does the whole sim-to-reality study in
  section 12.